In [62]:
library(DBI)
library(RPostgres)
library(tidyverse)
library(glue)
library(httr)
library(jsonlite)
library(dplyr)
library(tidyr)
library(lubridate)
library(stringr)
library(ggplot2)
library(readr)

source("Pitcher_Pitch_Selection_Static_Data.r")
source("Pitch_Selection_Multipliers.r")
source("Pitcher_First_Pitch_Prediction.r")
source("Pitch_Prediction.r")
source("Scouting_Report.r")


statcast_data <- read_csv("C:/Users/james.villegas/OneDrive - Rotork plc/Desktop/Baseball/statcast_25_26.csv", show_col_types = FALSE)

# clean data

statcast_data_cleaned <- statcast_data %>%
drop_na(
    pitch_type
    )

# re arrange 
statcast_data_ordered <- statcast_data_cleaned %>%
arrange(
        game_date,
        game_pk,
        at_bat_number,
        pitch_number
        )

# filter out <= 1000 pitches, not qualified pitchers

statcast_data_grouped <- statcast_data_ordered %>%
group_by(
    pitcher
    ) %>%
summarise(
    count = n()
    ) %>%
filter(
    count >= 1000
    ) %>%
select(
    pitcher
    )

# order data correctly to make sure in order

pitcher_id_list <- statcast_data_grouped$pitcher



New names:
• `` -> `...1`


In [98]:
# clean data

statcast_data_cleaned <- statcast_data %>%
drop_na(
    pitch_type
    )

# re arrange 
statcast_data_ordered <- statcast_data_cleaned %>%
arrange(
        game_date,
        game_pk,
        at_bat_number,
        pitch_number
        )

# filter out <= 1000 pitches, not qualified pitchers

statcast_data_grouped <- statcast_data_ordered %>%
group_by(
    pitcher
    ) %>%
summarise(
    count = n()
    ) %>%
filter(
    count >= 1000
    ) %>%
select(
    pitcher
    )

# order data correctly to make sure in order

pitcher_id_list <- statcast_data_grouped$pitcher
    

In [100]:
length(pitcher_id_list)

[1] 338

In [50]:
player_name <- pitcher_data$player_name[1]
player_id <- pitcher_data$pitcher[1]

pitcher_data <- pitcher_data %>%
drop_na(pitch_type)

# scout report
# grab all pitch types

pitch_totals <- pitcher_data %>%
    count(pitch_type, name = "total_pitch_count") %>%
    mutate(total_usage_perc = total_pitch_count / sum(total_pitch_count))

valid_pitches <- pitch_totals %>%
    filter(total_usage_perc >= 0.10) %>%
    pull(pitch_type)

pitcher_data <- pitcher_data %>%
    filter(pitch_type %in% valid_pitches)

pitch_types <- pitcher_data %>%
distinct(pitch_type)

pitcher_static_data <- pitcher_static_model(pitcher_data)

ball_count_list <- seq(0,3)
strike_count_list <- seq(0,2)
tto_count_list <- seq(1,2)
stance_list <- c('R', 'L')
runners_on_list <- c(FALSE, TRUE)
risp_list <- c(FALSE, TRUE)
prev_result_list <- c('B', 'S')
prev_pitch_list <- pitch_types$pitch_type

situation_row_list <- list()

for (tto in tto_count_list) {
    for (stance in stance_list) {
        for (runners in runners_on_list) {
            for (risp in risp_list) {
                for (ball in ball_count_list) {
                    for (strike in strike_count_list) {

                        # FIRST-PITCH CASE
                        if (ball == 0 && strike == 0) {

                            situation_row <- build_situation_row(
                                pitcher_data,
                                pitcher_static_data,
                                ball,
                                strike,
                                stance,
                                runners,
                                risp,
                                tto,
                                NA_character_,
                                NA_character_
                            )

                            situation_row_list[[length(situation_row_list) + 1]] <- situation_row
                            next   # ← THIS IS THE KEY
                        }

                        # ALL OTHER COUNTS
                        for (prev_pitch in prev_pitch_list) {
                            for (prev_result in prev_result_list) {

                                # Skip illegal states
                                if (ball > 0 && strike == 0 && prev_result == "S") next
                                if (ball == 0 && strike > 0 && prev_result == "B") next

                                situation_row <- build_situation_row(
                                    pitcher_data,
                                    pitcher_static_data,
                                    ball,
                                    strike,
                                    stance,
                                    runners,
                                    risp,
                                    tto,
                                    prev_pitch,
                                    prev_result
                                )

                                situation_row_list[[length(situation_row_list) + 1]] <- situation_row
                            }
                        }
                    }
                }
            }
        }
    }
}

scouting_report <- bind_rows(situation_row_list)

pitch_columns <- setdiff(names(scouting_report), c('balls', 'strikes', 'stance', 'runners_on', 'risp', 'tto', 'prev_pitch', 'prev_result'))

scouting_report_cleaned <- scouting_report %>%
filter(
    !if_all(all_of(pitch_columns), is.na)
    )

scouting_report_cleaned <- scouting_report_cleaned %>%
    mutate(across(all_of(pitch_columns), ~replace_na(.x, 0)))

scouting_report_cleaned <- scouting_report_cleaned %>%
mutate(
    pitcher_id = player_id,
    pitcher_name = player_name,
    situation_id = row_number()
)

scouting_report_cleaned <- scouting_report_cleaned %>%
relocate(
    pitcher_id, .before=balls
    ) %>%
relocate(
    pitcher_name, .after=pitcher_id
    ) %>%
relocate(
    situation_id, .after=pitcher_name
    )

final_scouting_report <- scouting_report_cleaned %>%
pivot_longer(
    pitch_columns,
    names_to = 'pitch_type',
    values_to = 'probability'
    )

pitcher_id,pitcher_name,situation_id,balls,strikes,stance,runners_on,risp,tto,prev_pitch,prev_result,pitch_type,probability
<dbl>,<chr>,<int>,<int>,<int>,<chr>,<lgl>,<lgl>,<int>,<chr>,<chr>,<chr>,<dbl>
808967,"Yamamoto, Yoshinobu",1041,0,2,L,TRUE,TRUE,2,FS,S,FS,90.48
808967,"Yamamoto, Yoshinobu",1058,1,2,L,TRUE,TRUE,2,FS,S,FS,89.45
808967,"Yamamoto, Yoshinobu",489,0,2,L,TRUE,TRUE,1,FS,S,FS,89.40
808967,"Yamamoto, Yoshinobu",1043,0,2,L,TRUE,TRUE,2,CU,S,FS,89.09
808967,"Yamamoto, Yoshinobu",506,1,2,L,TRUE,TRUE,1,FS,S,FS,88.23
808967,"Yamamoto, Yoshinobu",1044,0,2,L,TRUE,TRUE,2,FC,S,FS,88.22
808967,"Yamamoto, Yoshinobu",1062,1,2,L,TRUE,TRUE,2,CU,S,FS,87.90
808967,"Yamamoto, Yoshinobu",491,0,2,L,TRUE,TRUE,1,CU,S,FS,87.86
808967,"Yamamoto, Yoshinobu",765,0,2,R,TRUE,TRUE,2,FS,S,FS,87.38
